# OpenStudio Parametric Study Workflow

This notebook runs energy simulations for various building envelope improvement scenarios.


In [ ]:
# Standard library
import json
import os
import subprocess
import time
from itertools import product
import sys
from pathlib import Path
import configparser
import shutil

# Add OpenStudio 3.11.0 Python bindings to path BEFORE importing
OPENSTUDIO_PYTHON_PATH = "/Applications/OpenStudio-3.11.0/Python"
if OPENSTUDIO_PYTHON_PATH not in sys.path:
    sys.path.insert(0, OPENSTUDIO_PYTHON_PATH)

import pandas as pd
import openstudio

# Read EC3 API Token from config.ini
def get_ec3_api_token():
    """Read EC3 API token from config.ini file."""
    script_dir = Path.cwd()
    repo_root = script_dir.parent.parent
    config_path = repo_root / "config.ini"
    
    if not config_path.exists():
        print(f"Warning: config.ini not found at {config_path}")
        return None
    
    config = configparser.ConfigParser()
    config.read(config_path)
    
    try:
        return config["EC3_API_TOKEN"]["API_TOKEN"]
    except KeyError:
        print("Warning: EC3_API_TOKEN not found in config.ini")
        return None

EC3_API_TOKEN = get_ec3_api_token()


# =========================
# HELPER FUNCTIONS
# =========================

def get_city_weather_files(city_name, base_weather_path):
    """
    Get the EPW and DDY file paths for a given city name.
    """
    city_folder_path = os.path.join(base_weather_path, city_name)
    if not os.path.exists(city_folder_path):
        print(f"Error: Folder '{city_folder_path}' does not exist")
        return None

    epw_file = None
    ddy_file = None
    for filename in os.listdir(city_folder_path):
        if filename.lower().endswith(".epw"):
            epw_file = os.path.join(city_folder_path, filename)
        elif filename.lower().endswith(".ddy"):
            ddy_file = os.path.join(city_folder_path, filename)

    return {"epw": epw_file, "ddy": ddy_file}


def generate_scenario_name(scenario_dict):
    """
    Generate a unique scenario name from scenario parameters.
    
    Example: baseline_SmallOffice_Chicago
             wall_r20_SmallOffice_Chicago
             roof_r30_SmallOffice_Chicago
             door_wooden_SmallOffice_Chicago
             all_wall_r20_roof_r30_window_u0.25_door_wooden_SmallOffice_Chicago
    """
    parts = []
    
    if scenario_dict.get("is_baseline", False):
        parts.append("baseline")
    else:
        # Check if it's an "all measures" scenario
        measure_count = sum([
            1 if scenario_dict.get("wall_r_value") else 0,
            1 if scenario_dict.get("roof_r_value") else 0,
            1 if scenario_dict.get("window_u_factor") else 0,
            1 if scenario_dict.get("door_option") else 0,
        ])
        
        if measure_count > 1:
            parts.append("all")
        
        if scenario_dict.get("wall_r_value"):
            parts.append(f"wall_r{scenario_dict['wall_r_value']}")
        if scenario_dict.get("roof_r_value"):
            parts.append(f"roof_r{scenario_dict['roof_r_value']}")
        if scenario_dict.get("window_u_factor"):
            parts.append(f"window_u{scenario_dict['window_u_factor']}")
        if scenario_dict.get("door_option"):
            # Abbreviate door option for cleaner names
            door_abbrev = scenario_dict['door_option'].replace(' ', '_').replace('door', 'd')
            parts.append(f"door_{door_abbrev}")
    
    # Add building type and city
    parts.append(scenario_dict["building_type"])
    parts.append(scenario_dict["city"])
    
    return "_".join(parts)


def apply_python_measure(
    model,
    measure_folder,
    measure_class_name,
    arguments_dict
):
    """
    Apply a Python measure to an OSM model (following apply_measure.py pattern).
    
    Args:
        model: OpenStudio model object
        measure_folder: Path to folder containing measure.py
        measure_class_name: Name of the measure class
        arguments_dict: Dictionary of measure arguments
    
    Returns:
        True if successful, False otherwise
    """
    try:
        # Add measure folder to Python path
        measure_folder_str = str(measure_folder)
        if measure_folder_str not in sys.path:
            sys.path.insert(0, measure_folder_str)
        
        # Import the measure module
        import measure as measure_module
        import importlib
        importlib.reload(measure_module)
        
        measure_class = getattr(measure_module, measure_class_name)
        
        # Create runner and measure instance
        osw = openstudio.WorkflowJSON()
        runner = openstudio.measure.OSRunner(osw)
        measure = measure_class()
        
        # Setup arguments
        args = measure.arguments(model)
        arg_map = openstudio.measure.convertOSArgumentVectorToMap(args)
        
        def set_arg(name, value):
            if name in arg_map:
                arg = arg_map[name]
                arg.setValue(value)
                arg_map[name] = arg
        
        # Set all arguments from dictionary
        for arg_name, arg_value in arguments_dict.items():
            set_arg(arg_name, arg_value)
        
        # Run the measure
        result = measure.run(model, runner, arg_map)
        
        # Check result
        result_value = runner.result().value().valueName()
        if result_value != "Success":
            print(f"  Measure result: {result_value}")
            for error in runner.result().errors():
                print(f"    ERROR: {error.logMessage()}")
            if measure_folder_str in sys.path:
                sys.path.remove(measure_folder_str)
            return False
        
        # Clean up
        if measure_folder_str in sys.path:
            sys.path.remove(measure_folder_str)
        
        return True
        
    except Exception as e:
        print(f"  ERROR applying measure: {str(e)}")
        import traceback
        traceback.print_exc()
        if measure_folder_str in sys.path:
            sys.path.remove(measure_folder_str)
        return False


def apply_reporting_measure(model, sql_file_path, measure_dir_path):
    """
    Apply OperatingCostCarbonReportingMeasure after simulation completes.
    This reads the SQL file and attaches cost/emissions data to the model.
    
    Args:
        model: OpenStudio model object
        sql_file_path: Path to eplusout.sql file
        measure_dir_path: Path to measures directory
        
    Returns:
        True if successful, False otherwise
    """
    try:
        # Check if SQL file exists
        if not os.path.exists(sql_file_path):
            print(f"  ⚠️  SQL file not found: {sql_file_path}")
            return False
        
        # Attach SQL file to model
        sql_file = openstudio.SqlFile(openstudio.toPath(str(sql_file_path)))
        model.setSqlFile(sql_file)
        
        # Load reporting measure
        reporting_measure_folder = Path(measure_dir_path) / "OperatingCostCarbonReportingMeasure"
        if reporting_measure_folder not in sys.path:
            sys.path.insert(0, str(reporting_measure_folder))
        
        import measure as reporting_module
        import importlib
        importlib.reload(reporting_module)
        
        from measure import OperatingCostCarbonReport
        
        # Create runner and measure instance
        osw = openstudio.WorkflowJSON()
        runner = openstudio.measure.OSRunner(osw)
        runner.setLastEnergyPlusSqlFilePath(openstudio.toPath(str(sql_file_path)))
        runner.setLastOpenStudioModel(model)
        
        measure = OperatingCostCarbonReport()
        
        # Setup arguments (empty for this measure)
        args = measure.arguments(model)
        arg_map = openstudio.measure.convertOSArgumentVectorToMap(args)
        
        # Run the reporting measure
        result = measure.run(runner, arg_map)
        
        # Check result
        result_value = runner.result().value().valueName()
        if result_value != "Success":
            print(f"  Reporting measure result: {result_value}")
            for error in runner.result().errors():
                print(f"    ERROR: {error.logMessage()}")
            return False
        
        # Clean up
        if str(reporting_measure_folder) in sys.path:
            sys.path.remove(str(reporting_measure_folder))
        
        return True
        
    except Exception as e:
        print(f"  ERROR applying reporting measure: {str(e)}")
        import traceback
        traceback.print_exc()
        if str(reporting_measure_folder) in sys.path:
            sys.path.remove(str(reporting_measure_folder))
        return False


# =========================
# CORE: SINGLE SCENARIO CREATION/RUN
# =========================

def create_simulation(
    city,
    base_run_dir,
    measure_dir_path,
    base_weather_path,
    scenario_dict,
    overwrite_existing=False,
    building_type="SmallOffice",
    template="90.1-2013",
    climate_zone="ASHRAE 169-2013-5A",
    openstudio_path="openstudio",
):
    """
    Build and run ONE simulation for a given scenario using a four-phase approach:
    
    Phase 1: Create baseline model using create_DOE_prototype_building (Ruby measure via CLI)
    Phase 2: Apply Python model measures directly to the model in memory (wall, roof, window, door)
    Phase 3: Run EnergyPlus simulation using OpenStudio CLI
    Phase 4: Apply OperatingCostCarbonReportingMeasure to read SQL and attach cost/emissions data
    
    scenario_dict should contain:
        - is_baseline: bool
        - wall_r_value: float or None
        - roof_r_value: float or None
        - window_u_factor: float or None
        - door_option: str or None
        - city: str
        - building_type: str
    """

    # --- Weather for this city ---
    wf = get_city_weather_files(city, base_weather_path)
    if wf is None or wf["epw"] is None:
        print(f"❌ Weather files not found for {city}")
        return None

    epw_path = wf["epw"]

    # --- Generate scenario name ---
    scenario_name = generate_scenario_name(scenario_dict)
    
    # --- Scenario folder ---
    scenario_run_dir = os.path.join(base_run_dir, scenario_name)
    os.makedirs(scenario_run_dir, exist_ok=True)

    run_dir = os.path.join(scenario_run_dir, "run")
    sql_output_path = os.path.join(run_dir, "run", "eplusout.sql")

    if os.path.exists(sql_output_path) and not overwrite_existing:
        print(f"⏭️  Skipping {scenario_name} - simulation already exists")
        return scenario_name
    elif os.path.exists(sql_output_path) and overwrite_existing:
        print(f"♻️  Overwriting existing simulation for {scenario_name}")

    os.makedirs(run_dir, exist_ok=True)
    
    # ========================================================================
    # PHASE 1: Create baseline model using create_DOE_prototype_building
    # ========================================================================
    baseline_osw_path = os.path.join(run_dir, "baseline.osw")
    
    # Build OSW for baseline model creation
    baseline_steps = [
        {
            "measure_dir_name": "create_DOE_prototype_building",
            "name": "Create DOE Prototype Building",
            "arguments": {
                "building_type": building_type,
                "template": template,
                "climate_zone": climate_zone,
            },
        }
    ]
    
    baseline_osw = {
        "weather_file": epw_path,
        "file_paths": [base_weather_path],
        "measure_paths": [os.path.abspath(measure_dir_path)],
        "steps": baseline_steps,
        "name": f"{scenario_name}_baseline",
    }
    
    with open(baseline_osw_path, "w") as f:
        json.dump(baseline_osw, f, indent=2)
    
    # Run baseline model creation
    try:
        result = subprocess.run(
            [openstudio_path, "run", "-w", baseline_osw_path],
            check=False,  # OpenStudio returns 0 even on workflow failure
            capture_output=True,
            text=True,
            cwd=run_dir,
        )
        
        # Check if OpenStudio workflow actually succeeded
        out_osw_path = os.path.join(run_dir, "out.osw")
        if os.path.exists(out_osw_path):
            with open(out_osw_path, 'r') as f:
                out_osw = json.load(f)
                status = out_osw.get('completed_status', 'Unknown')
                if status != 'Success':
                    print(f"❌ {scenario_name} baseline workflow failed with status: {status}")
                    if result.stdout:
                        print("Last 500 chars of output:")
                        print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
                    return None
    except subprocess.SubprocessError as e:
        print(f"❌ {scenario_name} baseline model creation subprocess error: {e}")
        return None
    except Exception as e:
        print(f"❌ {scenario_name} unexpected error during baseline creation: {e}")
        return None
    
    # Find the generated baseline model
    baseline_model_path = os.path.join(run_dir, "run", "in.osm")
    if not os.path.exists(baseline_model_path):
        print(f"❌ {scenario_name} baseline model not found at {baseline_model_path}")
        print(f"   Expected after successful baseline OSW execution")
        # List what files do exist for debugging
        run_run_dir = os.path.join(run_dir, "run")
        if os.path.exists(run_run_dir):
            print(f"   Files in {run_run_dir}:")
            for item in os.listdir(run_run_dir)[:10]:  # Show first 10 items
                print(f"     - {item}")
        return None
    
    # Copy baseline model to safe location (simulation.osw will delete run/ directory)
    baseline_copy_path = os.path.join(run_dir, "baseline.osm")
    shutil.copy2(baseline_model_path, baseline_copy_path)
    
    # ========================================================================
    # PHASE 2: Apply Python model measures to baseline model
    # ========================================================================
    final_model_path = baseline_copy_path  # Use the safe copy
    
    if not scenario_dict.get("is_baseline", False):
        # Load the baseline model from safe copy
        translator = openstudio.osversion.VersionTranslator()
        model_path_os = openstudio.toPath(str(baseline_copy_path))
        loaded_model = translator.loadModel(model_path_os)
        
        if not loaded_model.is_initialized():
            print(f"❌ {scenario_name} failed to load baseline model")
            return None
        
        model = loaded_model.get()
        
        # Apply wall insulation measure
        if scenario_dict.get("wall_r_value"):
            wall_measure_folder = Path(measure_dir_path) / "IncreaseInsulationRValueForExteriorWalls"
            wall_args = {
                "r_value": float(scenario_dict["wall_r_value"]),
                "analysis_period": 30,
                "gwp_statistic": "median",
                "api_key": EC3_API_TOKEN or "",
                "insulation_material_type": "Blown Fiberglass",
                "insulation_material_lifetime": 30,
                "insulation_thermal_conductivity": 0.0,
                "insulation_material_density": 0.0,
            }
            print(f"  Applying wall insulation (R={scenario_dict['wall_r_value']})...")
            success = apply_python_measure(
                model,
                wall_measure_folder,
                "IncreaseInsulationRValueForExteriorWalls",
                wall_args
            )
            if not success:
                print(f"❌ {scenario_name} wall measure failed")
                del model
                return None
        
        # Apply roof insulation measure
        if scenario_dict.get("roof_r_value"):
            roof_measure_folder = Path(measure_dir_path) / "IncreaseInsulationRValueForRoofs"
            roof_args = {
                "r_value": float(scenario_dict["roof_r_value"]),
                "analysis_period": 30,
                "gwp_statistic": "median",
                "api_key": EC3_API_TOKEN or "",
                "insulation_material_type": "Blown Fiberglass",
                "insulation_material_lifetime": 30,
                "insulation_thermal_conductivity": 0.0,
                "insulation_material_density": 0.0,
            }
            print(f"  Applying roof insulation (R={scenario_dict['roof_r_value']})...")
            success = apply_python_measure(
                model,
                roof_measure_folder,
                "IncreaseInsulationRValueForRoofs",
                roof_args
            )
            if not success:
                print(f"❌ {scenario_name} roof measure failed")
                del model
                return None
        
        # Apply window enhancement measure
        if scenario_dict.get("window_u_factor"):
            if EC3_API_TOKEN is None:
                print(f"⚠️  Warning: EC3 API token not found, skipping window enhancement")
            else:
                window_measure_folder = Path(measure_dir_path) / "window_enhancement"
                
                # Map U-factor to glass pane configuration
                u_factor = float(scenario_dict["window_u_factor"])
                if u_factor >= 0.30:
                    num_panes = 2
                elif u_factor >= 0.25:
                    num_panes = 3
                else:
                    num_panes = 3
                
                window_args = {
                    "glass_option": "provide user_num_panes",
                    "user_num_panes": num_panes,
                    "space_infiltration_reduction_percent": 50.0,
                    "glass_pane_thickness": 0.003,
                    "gap_thickness": 0.013,
                    "glass_solar_transmittance": 0.7,
                    "glass_visible_transmittance": 0.8,
                    "glass_front_emissivity": 0.84,
                    "glass_back_emissivity": 0.84,
                    "glass_front_solar_reflectance": 0.15,
                    "glass_back_solar_reflectance": 0.15,
                    "glass_front_visible_reflectance": 0.1,
                    "glass_back_visible_reflectance": 0.1,
                    "analysis_period": 30,
                    "glass_lifetime": 15,
                    "wf_lifetime": 15,
                    "caulking_lifetime": 10,
                    "film_lifetime": 10,
                    "weatherstrip_lifetime": 10,
                    "wf_option": "none",
                    "caulking_option": "none",
                    "caulking_thickness": 0.003,
                    "film_option": "none",
                    "film_visible_transmittance": 0.0,
                    "film_solar_transmittance": 0.0,
                    "film_thermal_emissivity": 0.0,
                    "film_thermal_resistance": 0.0,
                    "weatherstrip_option": "none",
                    "length_per_unit": 1.0,  # Must be > 0
                    "secondary_glazing_option": "none",
                    "api_key": EC3_API_TOKEN,
                    "gwp_statistic": "median",
                }
                
                print(f"  Applying window enhancement (U={scenario_dict['window_u_factor']}, {num_panes} panes)...")
                success = apply_python_measure(
                    model,
                    window_measure_folder,
                    "WindowEnhancement",
                    window_args
                )
                if not success:
                    print(f"❌ {scenario_name} window measure failed")
                    del model
                    return None
        
        # Apply door enhancement measure
        if scenario_dict.get("door_option"):
            if EC3_API_TOKEN is None:
                print(f"⚠️  Warning: EC3 API token not found, skipping door enhancement")
            else:
                door_measure_folder = Path(measure_dir_path) / "door_enhancement"
                door_args = {
                    "space_infiltration_reduction_percent": 30.0,
                    "alter_coef": False,
                    "door_area_per_unit": 1.95,
                    "analysis_period": 30,
                    "door_bottom_seal_option": "automatic door bottom",
                    "door_top_side_seal_option": "jamb weatherstrip",
                    "door_option": scenario_dict["door_option"],
                    "strip_lifetime": 15,
                    "door_lifetime": 30,
                    "gwp_statistic": "median",
                    "api_key": EC3_API_TOKEN,
                    "length_per_unit_bottom_side": 0.9144,
                    "length_per_unit_other_sides": 5.1816,
                    "door_thermal_conductivity": 0.0,
                    "door_density": 0.0,
                    "door_thickness": 0.0,
                }
                
                print(f"  Applying door enhancement (door={scenario_dict['door_option']})...")
                success = apply_python_measure(
                    model,
                    door_measure_folder,
                    "DoorEnhancement",
                    door_args
                )
                if not success:
                    print(f"❌ {scenario_name} door measure failed")
                    del model
                    return None
        
        # Save modified model
        modified_model_path = os.path.join(run_dir, "modified.osm")
        model.save(openstudio.toPath(modified_model_path), True)
        final_model_path = modified_model_path
        del model
    
    # ========================================================================
    # PHASE 3: Run EnergyPlus simulation using OpenStudio CLI
    # ========================================================================
    simulation_osw_path = os.path.join(run_dir, "simulation.osw")
    
    # Create OSW for EnergyPlus simulation (no measures in steps array)
    simulation_osw = {
        "weather_file": epw_path,
        "seed_file": os.path.abspath(final_model_path),
        "file_paths": [base_weather_path],
        "measure_paths": [os.path.abspath(measure_dir_path)],
        "steps": [],  # Empty - just run simulation
        "name": scenario_name,
    }
    
    with open(simulation_osw_path, "w") as f:
        json.dump(simulation_osw, f, indent=2)
    
    # Run EnergyPlus simulation
    try:
        result = subprocess.run(
            [openstudio_path, "run", "-w", simulation_osw_path],
            check=True,
            capture_output=True,
            text=True,
            cwd=run_dir,
            timeout=600  # 10 minute timeout
        )
    except subprocess.TimeoutExpired:
        print(f"❌ {scenario_name} simulation timed out")
        return None
    except subprocess.CalledProcessError as e:
        print(f"❌ {scenario_name} simulation failed.")
        print("STDOUT:\n", e.stdout[-500:] if len(e.stdout) > 500 else e.stdout)
        print("STDERR:\n", e.stderr[-500:] if len(e.stderr) > 500 else e.stderr)
        return None
    
    # ========================================================================
    # PHASE 4: Apply OperatingCostCarbonReportingMeasure to read SQL results
    # ========================================================================
    sql_file_path = os.path.join(run_dir, "run", "eplusout.sql")
    
    if not os.path.exists(sql_file_path):
        print(f"❌ {scenario_name} SQL file not found after simulation")
        return None
    
    # Load the final model (either baseline or modified)
    translator = openstudio.osversion.VersionTranslator()
    translator.setAllowNewerVersions(True)
    model_path_os = openstudio.toPath(str(final_model_path))
    loaded_model = translator.loadModel(model_path_os)
    
    if not loaded_model.is_initialized():
        print(f"❌ {scenario_name} failed to load model for reporting")
        return None
    
    model = loaded_model.get()
    
    # Apply reporting measure
    print(f"  Applying reporting measure...")
    reporting_success = apply_reporting_measure(model, sql_file_path, measure_dir_path)
    
    if not reporting_success:
        print(f"⚠️  {scenario_name} reporting measure failed, but simulation completed")
        # Don't return None - simulation succeeded even if reporting failed
    
    # Save model with AdditionalProperties attached
    final_output_path = os.path.join(run_dir, "run", "in.osm")
    model.save(openstudio.toPath(final_output_path), True)
    del model
    
    print(f"✅ Completed: {scenario_name}")
    return scenario_name


# =========================
# SCENARIO GENERATION
# =========================

def generate_scenarios(
    cities,
    building_types,
    run_wall_insulation=False,
    run_roof_insulation=False,
    run_window_enhancement=False,
    run_door_enhancement=False,
    run_all_measures=False,
    wall_r_values=None,
    roof_r_values=None,
    window_u_factors=None,
    door_options=None,
):
    """
    Generate scenarios with simplified logic:
    - Always includes baseline
    - Individual measures (wall only, roof only, window only, door only)
    - All measures combined
    
    Returns a list of scenario dictionaries.
    """
    scenarios = []
    
    # 1) ALWAYS RUN BASELINE
    for city, building_type in product(cities, building_types):
        scenarios.append({
            "is_baseline": True,
            "city": city,
            "building_type": building_type,
            "wall_r_value": None,
            "roof_r_value": None,
            "window_u_factor": None,
            "door_option": None,
        })
    
    # 2) INDIVIDUAL MEASURES
    # Wall insulation only
    if run_wall_insulation and wall_r_values:
        for city, building_type, wall_r in product(cities, building_types, wall_r_values):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": wall_r,
                "roof_r_value": None,
                "window_u_factor": None,
                "door_option": None,
            })
    
    # Roof insulation only
    if run_roof_insulation and roof_r_values:
        for city, building_type, roof_r in product(cities, building_types, roof_r_values):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": None,
                "roof_r_value": roof_r,
                "window_u_factor": None,
                "door_option": None,
            })
    
    # Window enhancement only
    if run_window_enhancement and window_u_factors:
        for city, building_type, window_u in product(cities, building_types, window_u_factors):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": None,
                "roof_r_value": None,
                "window_u_factor": window_u,
                "door_option": None,
            })
    
    # Door enhancement only
    if run_door_enhancement and door_options:
        for city, building_type, door_opt in product(cities, building_types, door_options):
            scenarios.append({
                "is_baseline": False,
                "city": city,
                "building_type": building_type,
                "wall_r_value": None,
                "roof_r_value": None,
                "window_u_factor": None,
                "door_option": door_opt,
            })
    
    # 3) ALL MEASURES COMBINED
    if run_all_measures:
        walls = wall_r_values if wall_r_values else [None]
        roofs = roof_r_values if roof_r_values else [None]
        windows = window_u_factors if window_u_factors else [None]
        doors = door_options if door_options else [None]
        
        for city, building_type, wall_r, roof_r, window_u, door_opt in product(
            cities, building_types, walls, roofs, windows, doors
        ):
            if wall_r or roof_r or window_u or door_opt:
                scenarios.append({
                    "is_baseline": False,
                    "city": city,
                    "building_type": building_type,
                    "wall_r_value": wall_r,
                    "roof_r_value": roof_r,
                    "window_u_factor": window_u,
                    "door_option": door_opt,
                })
    
    return scenarios


# =========================
# POSTPROCESS: COLLECT RESULTS
# =========================

def extract_additional_properties_from_osm(osm_path):
    """
    Extract AdditionalProperties from an OSM file.
    
    Args:
        osm_path: Path to the OSM file
        
    Returns:
        Dictionary of property name: value pairs
    """
    try:
        translator = openstudio.osversion.VersionTranslator()
        translator.setAllowNewerVersions(True)
        model_path_os = openstudio.toPath(str(osm_path))
        loaded_model = translator.loadModel(model_path_os)
        
        if not loaded_model.is_initialized():
            return {}
        
        model = loaded_model.get()
        building = model.getBuilding()
        props = building.additionalProperties()
        
        # Extract all feature names and values
        prop_dict = {}
        feature_names = props.featureNames()
        for i in range(len(feature_names)):
            name = feature_names[i]
            value_variant = props.getFeatureAsString(name)
            if value_variant.is_initialized():
                value_str = value_variant.get()
                # Try to convert to numeric if possible
                try:
                    prop_dict[name] = float(value_str)
                except ValueError:
                    prop_dict[name] = value_str
        
        del model
        return prop_dict
        
    except Exception as e:
        print(f"  ⚠️  Failed to extract properties from {osm_path}: {e}")
        return {}


def collect_results_to_csv(base_run_dir, csv_base_name="parametric_results"):
    """
    Walk all run directories, extract AdditionalProperties from OSM files,
    and collect all results into a comprehensive CSV with:
    - Scenario names as rows (first column)
    - Measure-specific embodied carbon and costs as columns
    - Operating costs and emissions as columns
    """
    rows = []

    for scenario_folder in os.listdir(base_run_dir):
        scenario_path = os.path.join(base_run_dir, scenario_folder)
        if not os.path.isdir(scenario_path):
            continue
        
        # Look for SQL file
        sql_path = os.path.join(scenario_path, "run", "run", "eplusout.sql")
        if not os.path.exists(sql_path):
            continue
        
        # Parse scenario name
        scenario_parts = scenario_folder.split("_")
        building_type = scenario_parts[-2] if len(scenario_parts) >= 2 else "Unknown"
        city = scenario_parts[-1] if len(scenario_parts) >= 1 else "Unknown"
        
        is_baseline = scenario_parts[0] == "baseline"
        is_all_measures = scenario_parts[0] == "all" if not is_baseline else False
        wall_r = None
        roof_r = None
        window_u = None
        door_opt = None
        
        if not is_baseline:
            for part in scenario_parts:
                if part.startswith("wall"):
                    try:
                        wall_r = float(part.replace("wall_r", ""))
                    except:
                        pass
                elif part.startswith("roof"):
                    try:
                        roof_r = float(part.replace("roof_r", ""))
                    except:
                        pass
                elif part.startswith("window"):
                    try:
                        window_u = float(part.replace("window_u", ""))
                    except:
                        pass
                elif part.startswith("door"):
                    try:
                        # Extract door option from part
                        door_opt = part.replace("door_", "")
                    except:
                        pass
        
        row = {
            "scenario_name": scenario_folder,
            "city": city,
            "building_type": building_type,
            "is_baseline": is_baseline,
            "is_all_measures": is_all_measures,
            "wall_r_value": wall_r,
            "roof_r_value": roof_r,
            "window_u_factor": window_u,
            "door_option": door_opt,
        }
        
        # Extract AdditionalProperties from final OSM file
        osm_path = os.path.join(scenario_path, "run", "run", "in.osm")
        if os.path.exists(osm_path):
            print(f"  Extracting properties from {scenario_folder}...")
            additional_props = extract_additional_properties_from_osm(osm_path)
            
            # Organize properties by measure and type
            for key, value in additional_props.items():
                # Store all properties directly - they should already have measure prefixes
                row[key] = value
        
        rows.append(row)

    if not rows:
        print("\nℹ️ No simulation results found.")
        return

    df_results = pd.DataFrame(rows)
    
    # Reorder columns for better readability - organize by measure and metric type
    priority_cols = [
        "scenario_name", 
        "city", 
        "building_type", 
        "is_baseline",
        "is_all_measures",
        "wall_r_value", 
        "roof_r_value", 
        "window_u_factor",
        "door_option",
    ]
    
    # Group measure-specific embodied carbon columns
    embodied_carbon_cols = [col for col in df_results.columns if "total_additional_embodied_carbon_kg" in col]
    embodied_carbon_cols.sort()
    
    # Group measure-specific material cost columns
    material_cost_cols = [col for col in df_results.columns if "total_additional_material_cost_$" in col]
    material_cost_cols.sort()
    
    # Group measure-specific labour cost columns
    labour_cost_cols = [col for col in df_results.columns if "total_additional_labour_cost_$" in col]
    labour_cost_cols.sort()
    
    # Group measure-specific overhead/profit cost columns
    overhead_cost_cols = [col for col in df_results.columns if "total_additional_overhead_profit_cost_$" in col]
    overhead_cost_cols.sort()
    
    # Operating emissions columns
    operating_emissions_cols = [col for col in df_results.columns if "operating_emissions" in col]
    operating_emissions_cols.sort()
    
    # Operating cost columns
    operating_cost_cols = [col for col in df_results.columns if "annual_" in col and "cost" in col]
    operating_cost_cols.sort()
    
    # Build the final column order
    existing_priority = [col for col in priority_cols if col in df_results.columns]
    
    # Combine in logical order
    organized_cols = (
        existing_priority + 
        embodied_carbon_cols + 
        material_cost_cols + 
        labour_cost_cols +
        overhead_cost_cols +
        operating_emissions_cols + 
        operating_cost_cols
    )
    
    # Add any remaining columns not yet included
    remaining_cols = [col for col in df_results.columns if col not in organized_cols]
    remaining_cols.sort()
    column_order = organized_cols + remaining_cols
    
    df_results = df_results[column_order]
    
    csv_path = os.path.join(base_run_dir, f"{csv_base_name}.csv")
    df_results.to_csv(csv_path, index=False)
    print(f"\n🧾 Results CSV: {csv_path}")
    print(f"   Total scenarios: {len(df_results)}")
    print(f"   Columns: {len(df_results.columns)}")
    print(f"\n📊 Column organization:")
    print(f"   - Embodied carbon columns: {len(embodied_carbon_cols)}")
    print(f"   - Material cost columns: {len(material_cost_cols)}")
    print(f"   - Labour cost columns: {len(labour_cost_cols)}")
    print(f"   - Overhead/profit cost columns: {len(overhead_cost_cols)}")
    print(f"   - Operating emissions columns: {len(operating_emissions_cols)}")
    print(f"   - Operating cost columns: {len(operating_cost_cols)}")
    
    return df_results


# =========================
# GLOBAL SETTINGS
# =========================

# RUN NAME - change this for different parametric runs
RUN_NAME = "run_test_001"  # Change this for each new run
# OpenStudio executable path
OPENSTUDIO_PATH = "/Applications/OpenStudio-3.11.0/bin/openstudio"

OVERWRITE_EXISTING = False

# Paths
notebook_dir = Path.cwd()
base_weather_path = str(notebook_dir / "weather")
measure_dir_path = str(notebook_dir.parent / "measures")
base_run_dir = str(notebook_dir / "simulations" / RUN_NAME)  # Include RUN_NAME in path

# City climate zones
city_climate_zones = {
    "Amarillo":     "ASHRAE 169-2013-3B",
    # "Atlanta":      "ASHRAE 169-2013-3A",
    # "Baltimore":    "ASHRAE 169-2013-4A",
    # "Chicago":      "ASHRAE 169-2013-5A",
    # "Denver":       "ASHRAE 169-2013-5B",
    # "Duluth":       "ASHRAE 169-2013-7A",
    # "ElPaso":       "ASHRAE 169-2013-3B",
    # "Fairbanks":    "ASHRAE 169-2013-8A",
    # "Helena":       "ASHRAE 169-2013-6B",
    # "Houston":      "ASHRAE 169-2013-2A",
    # "Miami":        "ASHRAE 169-2013-1A",
    # "Minneapolis":  "ASHRAE 169-2013-6A",
    # "Phoenix":      "ASHRAE 169-2013-2B",
    # "PortAngeles":  "ASHRAE 169-2013-4C",
    # "Portland":     "ASHRAE 169-2013-4C",
    # "SanFrancisco": "ASHRAE 169-2013-3C",
}

# =========================
# PARAMETRIC STUDY CONFIGURATION
# =========================

# Individual measures
RUN_WALL_INSULATION = True
RUN_ROOF_INSULATION = True
RUN_WINDOW_ENHANCEMENT = True  # Now properly configured for WindowEnhancement measure
RUN_DOOR_ENHANCEMENT = True    # Door enhancement with weatherstripping and door replacement

# All measures combined
RUN_ALL_MEASURES = False

# Building parameters
CITIES = list(city_climate_zones.keys())

BUILDING_TYPES = [
    "SmallOffice",
    # "MediumOffice",
    # "LargeOffice",
    # "SmallHotel",
    # "LargeHotel",
    # "Warehouse",
    # "RetailStandalone",
    # "RetailStripmall",
    # "PrimarySchool",
    # "SecondarySchool",
]

TEMPLATE = "90.1-2010"

# === MEASURE PARAMETERS ===
# Wall insulation R-values to test (in h·ft²·°F/Btu)
WALL_R_VALUES = [
    # 13,  # Basic
    # 20,  # Good
    30,  # Excellent
]

# Roof insulation R-values to test (in h·ft²·°F/Btu)
ROOF_R_VALUES = [
    # 20,  # Basic
    # 30,  # Good
    40,  # Excellent
]

# Window U-factors to test (in Btu/h·ft²·°F)
WINDOW_U_FACTORS = [
    # 0.30,  # Double pane low-e
    # 0.25,  # Triple pane
    0.20,  # High performance triple pane
]

# Door options to test
DOOR_OPTIONS = [
    "wooden door",
    # "garage door",
    # "glass door",
    # "polystyrene core steel door",
    # "polyurethane core steel door",
    # "honeycomb core steel door",
    # "stiffened core steel door",
]

# =========================
# MAIN - RUN PARAMETRIC STUDY
# =========================

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("PARAMETRIC STUDY: BUILDING ENERGY EFFICIENCY MEASURES")
    print(f"Using OpenStudio: {OPENSTUDIO_PATH}")
    print(f"Run Name: {RUN_NAME}")
    print(f"Output Directory: {base_run_dir}")
    print("=" * 70)
    
    # Generate scenarios
    print("\n📋 Generating scenarios...")
    scenarios = generate_scenarios(
        cities=CITIES,
        building_types=BUILDING_TYPES,
        run_wall_insulation=RUN_WALL_INSULATION,
        run_roof_insulation=RUN_ROOF_INSULATION,
        run_window_enhancement=RUN_WINDOW_ENHANCEMENT,
        run_door_enhancement=RUN_DOOR_ENHANCEMENT,
        run_all_measures=RUN_ALL_MEASURES,
        wall_r_values=WALL_R_VALUES if RUN_WALL_INSULATION or RUN_ALL_MEASURES else None,
        roof_r_values=ROOF_R_VALUES if RUN_ROOF_INSULATION or RUN_ALL_MEASURES else None,
        window_u_factors=WINDOW_U_FACTORS if RUN_WINDOW_ENHANCEMENT or RUN_ALL_MEASURES else None,
        door_options=DOOR_OPTIONS if RUN_DOOR_ENHANCEMENT or RUN_ALL_MEASURES else None,
    )
    
    total_sims = len(scenarios)
    baseline_count = sum(1 for s in scenarios if s["is_baseline"])
    individual_wall = sum(1 for s in scenarios if not s["is_baseline"] and s["wall_r_value"] and not s["roof_r_value"] and not s["window_u_factor"] and not s["door_option"])
    individual_roof = sum(1 for s in scenarios if not s["is_baseline"] and s["roof_r_value"] and not s["wall_r_value"] and not s["window_u_factor"] and not s["door_option"])
    individual_window = sum(1 for s in scenarios if not s["is_baseline"] and s["window_u_factor"] and not s["wall_r_value"] and not s["roof_r_value"] and not s["door_option"])
    individual_door = sum(1 for s in scenarios if not s["is_baseline"] and s["door_option"] and not s["wall_r_value"] and not s["roof_r_value"] and not s["window_u_factor"])
    all_measures = sum(1 for s in scenarios if not s["is_baseline"] and sum([bool(s["wall_r_value"]), bool(s["roof_r_value"]), bool(s["window_u_factor"]), bool(s["door_option"])]) > 1)
    
    print(f"\n📦 Total scenarios: {total_sims}")
    print(f"   - Cities: {len(CITIES)}")
    print(f"   - Building Types: {len(BUILDING_TYPES)}")
    print(f"\n   Breakdown:")
    print(f"   - Baseline: {baseline_count}")
    print(f"   - Wall only: {individual_wall}")
    print(f"   - Roof only: {individual_roof}")
    print(f"   - Window only: {individual_window}")
    print(f"   - Door only: {individual_door}")
    print(f"   - All measures: {all_measures}")
    print("=" * 70)
    
    sim_count = 0
    start_time = time.time()
    successful_scenarios = []
    failed_scenarios = []
    
    # Run all scenarios
    for scenario in scenarios:
        sim_count += 1
        city = scenario["city"]
        building_type = scenario["building_type"]
        climate_zone = city_climate_zones.get(city, "ASHRAE 169-2013-5A")
        
        scenario_name = generate_scenario_name(scenario)
        print(f"\n[{sim_count}/{total_sims}] {scenario_name}")
        
        sim_start = time.time()
        
        result = create_simulation(
            city=city,
            base_run_dir=base_run_dir,
            measure_dir_path=measure_dir_path,
            base_weather_path=base_weather_path,
            scenario_dict=scenario,
            overwrite_existing=OVERWRITE_EXISTING,
            building_type=building_type,
            template=TEMPLATE,
            climate_zone=climate_zone,
            openstudio_path=OPENSTUDIO_PATH,
        )
        
        if result:
            successful_scenarios.append(result)
        else:
            failed_scenarios.append(scenario_name)
        
        sim_elapsed = time.time() - sim_start
        print(f"   ⏱️  Time: {sim_elapsed/60:.1f} min")
    
    total_elapsed = time.time() - start_time
    
    # Summary
    print("\n" + "=" * 70)
    print("SIMULATION SUMMARY")
    print("=" * 70)
    print(f"⏱️  Total time: {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f} hours)")
    print(f"✅ Successful: {len(successful_scenarios)}/{total_sims}")
    print(f"❌ Failed: {len(failed_scenarios)}/{total_sims}")
    
    if failed_scenarios:
        print("\nFailed scenarios:")
        for failed in failed_scenarios:
            print(f"  - {failed}")
    
    # Collect results
    print("\n" + "=" * 70)
    print("COLLECTING RESULTS FROM OSM FILES")
    print("=" * 70)
    collect_results_to_csv(base_run_dir=base_run_dir, csv_base_name="parametric_results")
    
    print("\n✅ Parametric study complete!")



PARAMETRIC STUDY: BUILDING ENERGY EFFICIENCY MEASURES
Using OpenStudio: /Applications/OpenStudio-3.11.0/bin/openstudio
Run Name: run_test_001
Output Directory: /Users/cbianchi/Documents/GitHub/openstudio-ee-gem/lib/parametric_run/simulations/run_test_001

📋 Generating scenarios...

📦 Total scenarios: 5
   - Cities: 1
   - Building Types: 1

   Breakdown:
   - Baseline: 1
   - Wall only: 1
   - Roof only: 1
   - Window only: 1
   - Door only: 1
   - All measures: 0

[1/5] baseline_SmallOffice_Amarillo
